In [128]:
import os
import pandas as pd
import re
import torch
from collections import defaultdict
from pathlib import Path
import numpy as np

def get_indiv_concepts(formula) -> set:
    concepts = set()
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.add(c[:end_idx])
    return concepts

def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    raw=[]
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        unit_concepts[unit].update(concepts)
        raw.extend(concepts)
    
    return unit_concepts, set(raw)

def build_binary_mask(neuron_mask, foundational_concept_list) -> torch.Tensor:
    num_neurons = len(neuron_mask)
    num_concepts = len(foundational_concept_list)

    # Step 1: Initialize tensor
    tensor = torch.zeros((num_neurons, num_concepts), dtype=torch.float32)

    # Step 2: Fill in ones
    for i, concepts in enumerate(neuron_mask.values()):
        for j, concept in enumerate(foundational_concept_list):
            if concept in concepts:
                tensor[i, j] = 1.0

    # Step 3: Compute row sums
    row_sums = tensor.sum(dim=1, keepdim=True)

    # Step 4: Normalize safely
    dist_tensor = torch.zeros_like(tensor)
    row_mask = (row_sums != 0).squeeze(1)  # True for rows with sum > 0
    dist_tensor[row_mask] = tensor[row_mask] #/ row_sums[row_mask]

    return dist_tensor



def get_neurons_for_cps(concepts, mapping):
    neurons = []
    for neuron, cps in mapping.items():
        for c in cps:
            if c in concepts:
                neurons.append(neuron)
                break
    return neurons

def get_all_cps_for_pi(folder):
    root_path = Path(folder)

    # Find all matching CSV files
    csv_pattern = 'Cluster*IOUS1024N.csv'
    csv_files = list(root_path.rglob(csv_pattern))
    
    concept_dict=defaultdict(set)
    s=set()
    for csv_file in csv_files:
        concepts = []
        csv_file = os.path.join(folder, csv_file)
        df = pd.read_csv(csv_file)
        for unit, formula in zip(df.unit, df.best_name):
            concept_dict[unit].update(get_indiv_concepts(formula))
        
     
        for u, c in concept_dict.items():
            s.update(c)
    return concept_dict

In [152]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

# NOTE: get_all_cps_for_pi, build_binary_mask must be importable/defined in your env


def get_indiv_concepts(formula) -> list:
    concepts = []
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', str(formula))
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.append(c[:end_idx])
    return concepts


def get_pruned_folders_sorted(expls_dir):
    """
    Scan expls_dir for *%Pruned folders, return sorted by sparsity ascending.
    Returns list of (sparsity_pct, folder_path).
    """
    folders = []
    for name in os.listdir(expls_dir):
        match = re.match(r'([0-9.]+)%Pruned', name)
        if match and Path(os.path.join(expls_dir, name)).is_dir():
            folders.append((float(match.group(1)), os.path.join(expls_dir, name)))
    folders.sort(key=lambda x: x[0])
    return folders


def get_all_concepts_in_folder(folder_path):
    """
    Collect all concepts from all IOUS1024N CSVs in a folder.
    Matches original get_foundationals logic: extend (not set) per formula.
    """
    concepts = []
    for csv_name in os.listdir(folder_path):
        if 'IOUS1024N' not in csv_name:
            continue
        try:
            df = pd.read_csv(os.path.join(folder_path, csv_name))
            for formula in df.best_name:
                concepts.extend(get_indiv_concepts(formula))
        except Exception as e:
            print(f"  Warning: could not read {csv_name}: {e}")
    return concepts


def get_neuron_concept_freqs(folder_path, concept_list):
    """
    Compute per-concept neuron frequencies using the same pipeline as the
    original code:
        get_all_cps_for_pi  →  build_binary_mask  →  mask.sum(dim=0)

    col_sums[i] = number of neurons that express concept_list[i].
    """
    neuron_formulas = get_all_cps_for_pi(folder_path)
    mask = build_binary_mask(neuron_formulas, concept_list)
    col_sums = mask.sum(dim=0).numpy()

    freqs = {concept_list[i]: float(col_sums[i]) for i in range(len(concept_list))}
    return freqs


def build_folder_list_for_run(expls_dir, baseline_expls_dir):
    """
    Build the sorted folder list for a run, but always substitute the
    0.0%Pruned folder from baseline_expls_dir (the _5 run).
    All other sparsity levels come from expls_dir's own folders.
    """
    folders = get_pruned_folders_sorted(expls_dir)
    if not folders:
        return []

    # Find 0.0%Pruned in the baseline dir
    baseline_zero = None
    for pct, path in get_pruned_folders_sorted(baseline_expls_dir):
        if pct == 0.0:
            baseline_zero = (pct, path)
            break

    if baseline_zero is None:
        print(f"  Warning: no 0.0%Pruned folder in baseline {baseline_expls_dir}, using run's own.")
        return folders

    result = []
    replaced = False
    for pct, path in folders:
        if pct == 0.0:
            result.append(baseline_zero)  # always use _5's dense baseline
            replaced = True
        else:
            result.append((pct, path))

    if not replaced:
        result = [baseline_zero] + result  # run had no 0.0%Pruned, prepend baseline's

    return result


def get_foundationals_for_run(expls_dir, baseline_expls_dir):
    """
    Concepts present in every %Pruned folder for a single run.
    The 0.0%Pruned folder always comes from baseline_expls_dir.
    Matches original get_foundationals logic exactly.
    """
    folders = build_folder_list_for_run(expls_dir, baseline_expls_dir)
    if not folders:
        return set(), []

    per_folder = []
    for _, folder_path in folders:
        concepts = get_all_concepts_in_folder(folder_path)
        if concepts:
            per_folder.append(set(concepts))

    if not per_folder:
        return set(), folders

    return set.intersection(*per_folder), folders


def analyze_foundationals_across_runs(run_expls_dirs, baseline_expls_dir):
    """
    Analyze foundational concepts across multiple runs.

    - 0.0%Pruned (dense baseline) ALWAYS comes from baseline_expls_dir (_5).
    - All other sparsity levels come from each run's own folders.
    - Folders aligned by POSITION so slight name differences (57.65 vs 57.66)
      don't cause misalignment.
    - Freq counting uses get_all_cps_for_pi + build_binary_mask + col_sums,
      exactly matching the original get_topk_concepts pipeline.

    Returns:
        results: dict {
            sparsity_idx (1-based): {
                'pct_per_run':   [float, ...],
                'n_runs':        int,
                'avg_mean_freq': float,
                'freqs':         {concept: avg_freq},   # sorted descending
                'std':           {concept: std_freq},
            }
        }
        foundationals: list[str]
    """
    valid_dirs = [d for d in run_expls_dirs if Path(d).exists()]
    if not valid_dirs:
        print("No valid run directories found.")
        return {}, []

    if not Path(baseline_expls_dir).exists():
        print(f"Baseline directory not found: {baseline_expls_dir}")
        return {}, []

    # ── Step 1: foundationals = intersection across all runs ──────────────
    all_foundationals = []
    run_folders = []

    for run_dir in valid_dirs:
        f, folders = get_foundationals_for_run(run_dir, baseline_expls_dir)
        if f:
            all_foundationals.append(f)
            run_folders.append(folders)
            label = "(using own 0%)" if Path(run_dir) == Path(baseline_expls_dir) \
                    else f"(0% from {Path(baseline_expls_dir).parent.name})"
            print(f"  ✓ {Path(run_dir).parent.name}: {len(f)} foundational concepts {label}")
        else:
            print(f"  Warning: no foundational concepts for {run_dir}, skipping.")

    if not all_foundationals:
        print("No foundational concepts found.")
        return {}, []

    foundationals = list(set.intersection(*all_foundationals))
    print(f"\nFinal foundational concepts: {len(foundationals)} "
          f"(intersection across all {len(run_folders)} run(s))")

    # ── Step 2: average freqs across runs per positional level ────────────
    n_levels = max(len(f) for f in run_folders)
    results = {}

    for level_idx in range(n_levels):
        sparsity_idx = level_idx + 1  # 1-based

        run_freqs = []
        pct_per_run = []

        for run_i, folders in enumerate(run_folders):
            if level_idx >= len(folders):
                print(f"  Run {run_i+1} missing level {sparsity_idx}, skipping.")
                continue

            sparsity_pct, folder_path = folders[level_idx]
            pct_per_run.append(sparsity_pct)

            try:
                freqs = get_neuron_concept_freqs(folder_path, foundationals)
                run_freqs.append(freqs)
            except Exception as e:
                print(f"  Error computing freqs for {folder_path}: {e}")

        if not run_freqs:
            continue

        avg_freqs = {}
        std_freqs = {}
        for concept in foundationals:
            vals = [f.get(concept, 0.0) for f in run_freqs]
            avg_freqs[concept] = float(np.mean(vals))
            std_freqs[concept] = float(np.std(vals))

        # sort descending by avg freq
        avg_freqs = dict(sorted(avg_freqs.items(), key=lambda x: x[1], reverse=True))

        results[sparsity_idx] = {
            'pct_per_run':   pct_per_run,
            'n_runs':        len(run_freqs),
            'avg_mean_freq': float(np.mean(list(avg_freqs.values()))),
            'freqs':         avg_freqs,
            'std':           std_freqs,
        }

    return results, foundationals


def print_foundational_results(results, top_n=15):
    """Print top-N concepts per sparsity level with avg ± std."""
    for sparsity_idx in sorted(results.keys()):
        r = results[sparsity_idx]
        pct_str = ', '.join(f'{p:.3f}' for p in r['pct_per_run'])
        print(f"\n{'='*72}")
        print(f"  Sparsity level {sparsity_idx}  |  actual %: [{pct_str}]  |  runs: {r['n_runs']}")
        print(f"  avg mean freq: {r['avg_mean_freq']:.2f}")
        print(f"{'='*72}")
        print(f"  {'Concept':<42} {'Avg Freq':>10}  {'±Std':>8}")
        print(f"  {'-'*42} {'-'*10}  {'-'*8}")
        for i, (concept, avg_freq) in enumerate(r['freqs'].items()):
            if i >= top_n:
                break
            std = r['std'].get(concept, 0.0)
            print(f"  {concept:<42} {avg_freq:>10.1f}  {std:>8.2f}")


# ── Entry point ───────────────────────────────────────────────────────────

if __name__ == "__main__":
    model  = 'BERT'
    method = 'lottery_ticket'
    runs   = ['Run0.25_5', 'Run0.25_6', 'Run0.25_7']
    base   = f"/workspace/CCE_NLI/{model.upper()}/exp/{method}/{{run}}/Expls"

    run_expls_dirs     = [base.format(run=run) for run in runs]
    baseline_expls_dir = base.format(run='Run0.25_5')  # 0.0%Pruned always from _5

    results, foundationals = analyze_foundationals_across_runs(
        run_expls_dirs=run_expls_dirs,
        baseline_expls_dir=baseline_expls_dir,
    )
    print_foundational_results(results, top_n=15)

  ✓ Run0.25_5: 286 foundational concepts (using own 0%)
  ✓ Run0.25_6: 272 foundational concepts (0% from Run0.25_5)
  ✓ Run0.25_7: 273 foundational concepts (0% from Run0.25_5)

Final foundational concepts: 231 (intersection across all 3 run(s))

  Sparsity level 1  |  actual %: [0.000, 0.000, 0.000]  |  runs: 3
  avg mean freq: 29.52
  Concept                                      Avg Freq      ±Std
  ------------------------------------------ ----------  --------
  pre:tag:nn                                      547.0      0.00
  oth:overlap:overlap25                           303.0      0.00
  hyp:tag:nn                                      287.0      0.00
  oth:overlap:overlap50                           204.0      0.00
  pre:tag:jj                                      161.0      0.00
  hyp:tok:outside                                 158.0      0.00
  pre:tag:.                                       158.0      0.00
  hyp:tag:dt                                      155.0      0.00
  

In [148]:
def get_foundationals(root_dir):
    root_path = Path(root_dir)
    # Find all matching CSV files
    fldr_pattern = '*%Pruned'
    fldr_files = list(root_path.rglob(fldr_pattern))
    cross_iter_concept_dict=defaultdict(list)
    for fldr_file in fldr_files:
        concepts = []
        for csvs in os.listdir(os.path.join(root_dir, fldr_file)):
            if 'IOUS1024N' not in csvs: continue
            csv_file = os.path.join(root_dir, fldr_file, csvs)
            df = pd.read_csv(csv_file)
            for unit, formula in zip(df.unit, df.best_name):
                concepts.extend(get_indiv_concepts(formula))
        cross_iter_concept_dict[fldr_file]=set(concepts)
    preserved_concepts = set.intersection(*cross_iter_concept_dict.values())
    return list(preserved_concepts)
def get_topk_concepts(mask,k, concept_list):
    """Test if concepts are uniformly distributed across neurons."""
    col_sums = mask.sum(dim=0).numpy()
    sorted_idx = np.argsort(col_sums)[::-1]
    top=[]
    freqs={}
    
    for i in range(len(sorted_idx)):
        idx = sorted_idx[i]
        top.append(concept_list[idx])
        freqs[concept_list[idx]]=col_sums[idx]
    return top, sum(list(freqs.values()))/len(freqs), freqs
root_dir='/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_/Expls'
foundationals = get_foundationals(root_dir)
for direct in sorted(os.listdir(root_dir)):
    neuron_formulas = get_all_cps_for_pi(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{direct}')
    mask = build_binary_mask(neuron_formulas, foundationals)
    top,f,freqs=get_topk_concepts(mask,len(foundationals), foundationals)
    print(direct,f)
    print(freqs)
'''
'pre:tag:nn': 539.0,
 'hyp:tag:nn': 327.0,
 'oth:overlap:overlap25': 307.0,
 'oth:overlap:overlap50': 202.0,
 'pre:tag:jj': 178.0,
 'hyp:tok:outside': 149.0,
 'hyp:tok:sleeping': 145.0,
 'hyp:tag:in': 134.0,
 'hyp:tag:vbg': 133.0,
 'hyp:tag:dt': 133.0,
 'pre:tag:.': 133.0,
 'hyp:tok:for': 131.0,
'''


25.0%Pruned 23.899653979238753
{'pre:tag:nn': 550.0, 'hyp:tag:nn': 315.0, 'oth:overlap:overlap25': 314.0, 'oth:overlap:overlap50': 179.0, 'pre:tag:jj': 168.0, 'hyp:tok:outside': 142.0, 'hyp:tag:dt': 135.0, 'pre:tag:.': 129.0, 'hyp:tok:sleeping': 128.0, 'hyp:tag:in': 126.0, 'hyp:tag:vbg': 119.0, 'hyp:tok:to': 115.0, 'oth:overlap:overlap75': 113.0, 'pre:tag:in': 113.0, 'pre:tok:man': 113.0, 'hyp:tok:for': 111.0, 'hyp:tok:there': 107.0, 'pre:tag:dt': 101.0, 'hyp:tok:outdoors': 91.0, 'hyp:tok:eating': 89.0, 'pre:tok:woman': 88.0, 'hyp:tag:prp$': 88.0, 'hyp:tok:woman': 85.0, 'hyp:tok:swimming': 74.0, 'hyp:tok:people': 72.0, 'hyp:tok:tall': 69.0, 'pre:tok:girl': 68.0, 'hyp:tok:man': 62.0, 'hyp:tag:ex': 59.0, 'hyp:tok:playing': 59.0, 'pre:tok:sitting': 58.0, 'hyp:tok:sitting': 57.0, 'hyp:tok:wearing': 57.0, 'pre:tok:dog': 54.0, 'hyp:tag:jj': 54.0, 'pre:tok:blue': 52.0, 'hyp:tok:nobody': 51.0, 'hyp:tok:inside': 47.0, 'pre:tok:boy': 47.0, 'hyp:tag:.': 43.0, 'hyp:tok:human': 42.0, 'pre:tok:peopl

"\n'pre:tag:nn': 539.0,\n 'hyp:tag:nn': 327.0,\n 'oth:overlap:overlap25': 307.0,\n 'oth:overlap:overlap50': 202.0,\n 'pre:tag:jj': 178.0,\n 'hyp:tok:outside': 149.0,\n 'hyp:tok:sleeping': 145.0,\n 'hyp:tag:in': 134.0,\n 'hyp:tag:vbg': 133.0,\n 'hyp:tag:dt': 133.0,\n 'pre:tag:.': 133.0,\n 'hyp:tok:for': 131.0,\n"

In [106]:
from collections import Counter
from itertools import chain

def get_nonfoundational_freq(neuron_formulas, foundationals):
    # flatten all concepts
    all_concepts = chain.from_iterable(neuron_formulas.values())

    # keep only non-foundationals
    non_foundationals = [c for c in all_concepts if c not in foundationals]

    # count frequency
    return Counter(non_foundationals)
from collections import Counter
from itertools import chain

def get_all_concept_freq(neuron_formulas):
    a=[]
    for u, co in neuron_formulas.items():
        for c in co:
            a.append(c)
    
    return Counter(a)

In [107]:
def get_pi(model, method, run):
    """
    Collect sparsity values (the number before '%Pruned')
    from directory names inside:

    /workspace/CCE_NLI/{MODEL}/exp/{method}/{run}/Expls

    Returns:
        Sorted list of floats.
    """
    base_path = f"/workspace/CCE_NLI/{model.upper()}/exp/{method}/{run}/Expls"

    if not os.path.exists(base_path):
        raise FileNotFoundError(f"Path not found: {base_path}")

    sparsities = []

    for name in os.listdir(base_path):
        match = re.match(r"([0-9.]+)%Pruned", name)
        if match:
            sparsities.append(float(match.group(1)))
    if 0.0 not in sparsities:
        sparsities.insert(0,0.0)
    return sorted(sparsities)

run='Run0.25_5'
pis = get_pi('BERT', 'lottery_ticket', run)
print(pis)
for i,pi in enumerate(pis):
    if pi == 0.0:
        s=[]
        neuron_formulas = get_all_cps_for_pi(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{pi}%Pruned')
        for u in neuron_formulas:
            for i in neuron_formulas[u]:
                s.append(i)
        print("ubique ", len(s))
    else:
        neuron_formulas = get_all_cps_for_pi(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/{run}/Expls/{pi}%Pruned')
        
    mask = build_binary_mask(neuron_formulas, foundationals)
    top,f,_=get_topk_concepts(mask,len(foundationals), foundationals)
    print(f"Avg number of times a foundational concept is repeated at sparsity {pi}%: {f}")
    nonfound=get_nonfoundational_freq(neuron_formulas, foundationals)
    print(f"sum of nonfound frewq : {sum(list(nonfound.values()))}. num found: {len(nonfound)}, percent:{sum(list(nonfound.values()))/len(nonfound)}")
    
    all_freq = get_all_concept_freq(neuron_formulas)
    total = sum(all_freq.values())
    unique = len(all_freq)

    print(
        f"All concepts total freq: {total}, "
        f"unique: {unique}, "
        f"avg repetition: {total / unique if unique else 0}"
    )


[0.0, 25.0, 43.75, 57.812, 68.359, 76.27]
IN  /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster1IOUS1024N.csv
/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster1IOUS1024N.csv 2183
IN  /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster2IOUS1024N.csv
/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster2IOUS1024N.csv 4295
IN  /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster3IOUS1024N.csv
/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster3IOUS1024N.csv 7279
ubique  7279
sum of freqs : 5881.0. num found: 151
Avg number of times a foundational concept is repeated at sparsity 0.0%: 38.94701986754967
sum of nonfound frewq : 139

In [91]:
for i,pi in enumerate(['0.0', '25.0', '43.75', '57.812', '68.359', '76.27']):
    gen=0
    unit, unique1 = load_csv_data(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{pi}%Pruned/Cluster1IOUS1024N.csv')
    unit, unique2 = load_csv_data(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{pi}%Pruned/Cluster2IOUS1024N.csv')
    unit, unique3 = load_csv_data(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{pi}%Pruned/Cluster3IOUS1024N.csv')
    unique=unique1|unique2|unique3
    for cp in unique:
        if any(kw in cp for kw in [':tag:', 'oth:overlap']):
            gen += 1
    print(pi, gen/len(unique), gen, len(unique))

0.0 0.09368635437881874 46 491
25.0 0.09484536082474226 46 485
43.75 0.09270216962524655 47 507
57.812 0.09021113243761997 47 521
68.359 0.09325396825396826 47 504
76.27 0.0874751491053678 44 503


In [66]:
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/43.75%Pruned')
mask = build_binary_mask(neuron_formulas, foundationals)

top,f=get_topk_concepts(mask,len(foundationals), foundationals)
f

25.045283018867924

In [74]:
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/57.812%Pruned')
mask = build_binary_mask(neuron_formulas, foundationals)

top,f=get_topk_concepts(mask,len(foundationals), foundationals)
f

25.026415094339622

In [75]:
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/68.359%Pruned')
mask = build_binary_mask(neuron_formulas, foundationals)

top,f=get_topk_concepts(mask,len(foundationals), foundationals)
f

25.713207547169812

In [76]:
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/76.27%Pruned')
mask = build_binary_mask(neuron_formulas, foundationals)

top,f=get_topk_concepts(mask,len(foundationals), foundationals)
f

25.00377358490566

In [71]:
def get_non_foundationals(foundational, pi):
    unit_to_cp_dict = get_all_cps_for_pi(pi)
    allcps = set()
    for unit,cps in unit_to_cp_dict.items():
        allcps.update(cps)
    return list(allcps - set(foundational))
def get_topk_concepts(mask,k, concept_list):
    """Test if concepts are uniformly distributed across neurons."""
    col_sums = mask.sum(dim=0).numpy()
    sorted_idx = np.argsort(col_sums)[::-1]
    top=[]
    freqs={}
    for i in range(len(sorted_idx))[:k]:
        idx = sorted_idx[i]
        top.append(concept_list[idx])
        freqs[concept_list[idx]]=col_sums[idx]
    return top, sum(list(freqs.values()))/len(freqs)
root_dir='/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls'
foundationals = get_foundationals(root_dir)
get_non_foundationals(foundationals,'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/25.0%Pruned' )
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/25.0%Pruned')
mask = build_binary_mask(neuron_formulas, non_foundationals)
_,f=get_topk_concepts(mask,len(non_foundationals), non_foundationals)
f

0.7145390070921985

In [70]:
get_non_foundationals(foundationals,'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/43.75%Pruned' )
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/43.75%Pruned')
mask = build_binary_mask(neuron_formulas, non_foundationals)
_,f=get_topk_concepts(mask,len(non_foundationals), non_foundationals)
f

0.6968085106382979